# 01 — Exploratory Data Analysis & Preprocessing
**Multilingual Health QA (HASH / Zindi)** · maps to Rubric criterion 3 (Data/EDA, 10 pts).

This notebook profiles the corpus and justifies every preprocessing decision. All heavy
lifting lives in `src/` (imported below) so the analysis is reproducible from one command
(`python -m src.eda`) and never drifts from the training/inference code.

**Guiding insight (see `PLAN.md`):** the leaderboard is `0.37·ROUGE-1 + 0.37·ROUGE-L +
0.26·LLM-judge` — **74% is pure lexical overlap**. So EDA centers on the two levers that
move lexical overlap: *(1) emitting the correct language & script*, and *(2) reusing the
reference vocabulary/length*.


In [ ]:
# Colab setup — clone repo + install. On local run, skip the clone.
import sys, os
if 'google.colab' in sys.modules:
    !git clone -q https://github.com/<your-user>/multilingual_health_qa.git
    %cd multilingual_health_qa
    !pip install -q rouge-score scikit-learn matplotlib
# Ensure repo root on path
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

In [ ]:
import pandas as pd, numpy as np
from src import data as D
from src.preprocess import geez_ratio, latin_ratio, normalize_text

train = D.load_split('train', clean=True, prefix=False)
val   = D.load_split('val',   clean=True, prefix=False)
test  = D.load_split('test',  clean=True, prefix=False)
print(train.shape, val.shape, test.shape)
train.head(2)

## 1. Splits, columns, integrity
We verify row counts, nulls, and **leakage** across splits (a contaminated split would
make our offline ROUGE optimistic).

In [ ]:
print('nulls (train):', train.isna().sum().to_dict())
print('leakage report:', D.leakage_report(train, val, test))

**Finding.** Train 29,815 / Val 6,686 / Test 2,618. No nulls. **Zero exact-input
overlap** train↔val and train↔test → our Val ROUGE is an honest proxy for the leaderboard.
Note though: **1,482 duplicate inputs** and **11,750 duplicate outputs (~39%)** *within*
train — answers are heavily templated (analyzed in §4).

## 2. Subset / language distribution
8 subsets of the form `<Lang>_<Country>`. The test set mirrors train proportions, so a
single jointly-trained model is the right structure — but low-resource languages
(Amharic, Swahili, Luganda, Akan) are small and need explicit attention.

In [ ]:
disp = pd.DataFrame({'train': train.subset.value_counts(),
                     'test':  test.subset.value_counts()})
disp['train_%'] = (disp.train / disp.train.sum() * 100).round(1)
disp['test_%']  = (disp.test  / disp.test.sum()  * 100).round(1)
disp.loc[:, ['language','script']] = train.groupby('subset')[['language','script']].first()
disp

![subset distribution](../reports/figures/01_subset_distribution.png)

**Finding.** ~61% of train is English. The four target African languages total ~39%; **Amharic is the smallest (1,845 rows) and the only Ge'ez-script language** — our hardest case.

## 3. Length distributions → truncation budgets
Decides `max_input_len` and generation `max_length` (a direct ROUGE lever — truncated
answers lose recall).

In [ ]:
for col in ['input','output']:
    w = train[col].fillna('').str.split().map(len)
    print(f'{col:7s} words  median={w.median():.0f}  p90={w.quantile(.9):.0f}  p99={w.quantile(.99):.0f}  max={w.max()}')

![lengths](../reports/figures/02_length_distributions.png)
![lengths by subset](../reports/figures/03_output_length_by_subset.png)

**Decision.** Inputs: median 13 words, max 83 → **`max_input_len=128` tokens** covers ~all. Outputs: median 61, p90 153 words → **generation `max_length=256` tokens** (≈ p90) balances recall vs. runaway repetition. We log this choice as a tunable in Phase 3.

## 4. Script handling (Ge'ez vs Latin) + duplication
The single most important multilingual check: Amharic answers must be **Ge'ez script**.
A model that emits Latin/English for Amharic scores ~0 on ROUGE there.

In [ ]:
rows = []
for s in D.SUBSETS:
    o = train[train.subset==s].output
    rows.append({'subset': s, 'geez': o.map(geez_ratio).mean().round(3),
                 'latin': o.map(latin_ratio).mean().round(3)})
pd.DataFrame(rows)

![script composition](../reports/figures/04_script_composition.png)

**Finding.** Amharic answers are **96.6% Ge'ez**; every other subset is ~100% Latin. → We (a) keep Unicode **NFC** normalization so Ge'ez sequences tokenize canonically, (b) pick a tokenizer with Ge'ez coverage (mT5/NLLB do), and (c) add a *wrong-language guard* at inference that flags Amharic predictions which aren't Ge'ez (`src/infer.py`).

In [ ]:
vc = train.output.value_counts()
print(f'unique answers: {vc.size:,} / {len(train):,} rows  ->  {1-vc.size/len(train):.1%} are duplicates')
vc.head(5)

![duplicate outputs](../reports/figures/05_duplicate_outputs.png)

**Finding & implication.** ~39% of answers are repeated templates. This is *why a retrieval baseline is strong* (copying a real template often matches the reference) and *why we must dedup carefully*: we keep duplicates in training (they reflect the true answer distribution the metric rewards) but watch for them inflating apparent val scores.

## 5. Question→answer lexical reuse
How much of the answer vocabulary is already in the question? Informs whether copying /
extraction helps.

In [ ]:
s = train.sample(4000, random_state=42)
rec = [len(set(str(q).lower().split()) & set(str(a).lower().split())) / max(1,len(set(str(q).lower().split())))
       for q,a in zip(s.input, s.output)]
print('mean question-word recall in answer:', round(float(np.mean(rec)),3))

![qa overlap](../reports/figures/06_qa_vocab_overlap.png)
![input vs output](../reports/figures/08_input_vs_output_length.png)

**Finding.** ~40% of question words reappear in the answer, but input/output lengths are uncorrelated — answers are elaborations, not echoes. Pure extraction won't suffice; we need generation conditioned on the question + language tag.

## 6. Preprocessing decisions (summary)
| Decision | Choice | Why |
|---|---|---|
| Unicode | **NFC** | canonical Ge'ez/Latin → stable tokenization & ROUGE counts |
| Whitespace | collapse + strip | removes scrape artifacts that fragment tokens |
| Case / punctuation | **keep** | ROUGE scores refs as-written; Ge'ez has no case |
| Invisible/control chars | strip | zero-width & BOM inflate token counts |
| Task prefix | `"<subset>: <q>"` | steers language + script with one shared model |
| `max_input_len` | 128 | covers max 83-word inputs |
| gen `max_length` | 256 | ≈ p90 output length |
All implemented in `src/preprocess.py` + `src/data.py`.

## 7. Baseline anchor (B0) — TF-IDF retrieval
A GPU-free floor: for each question, return the nearest training answer (per subset).
Scored with our leaderboard-mirroring ROUGE (`src/metrics.py`).

In [ ]:
from src.baseline_tfidf import run
res = run()
pd.DataFrame(res['by_lang']).T.round(4)

![baseline rouge](../reports/figures/07_baseline_rouge_by_language.png)

**B0 result: Val proxy ≈ 0.394** (R1 0.421 / RL 0.366). Per-language: easy = Swahili/Eng-Kenya (~0.58), **hard = Amharic (0.14)** and Ghanaian subsets (Akan/Eng-Gha ~0.22). This sets the bar the fine-tuned seq2seq model must beat, and tells us *where* to spend effort (Amharic, Akan).

## Next
→ `notebooks/colab_run.ipynb` fine-tunes mT5/NLLB end-to-end on a GPU and logs results to
`experiments/`. EDA findings above drive its config (prefix, lengths, low-resource upsampling).